In [1]:
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
import scipy.optimize
import datetime
import sys

class gel_3D:
    def __init__(self, length=90.0, width =15.0, thickness=1.6, phi0=0.2, mu_bar=-0.3):
        self.phi0 = phi0
        print(f'Initial polymer volume fraction = {phi0:.2f} [dimensionless];', end=' ')
        self.mu_bar = mu_bar   # 🔥 NUEVO
        print(f'Normalized chemical potential = {mu_bar:.6f} [dimensionless?];', end=' ')
        #
        # Values for T and V_m taken from p.1584 in Kang & Huang, JMPS 58 (2010)
        T = 25+273.15   # 25ºC, in [K]
        K_B = 1.380649e-23  # in m^2*kg*s^{-2}*K^{-1}, i.e. in N*m/K
        V_m = 3e-29     # volume of a molecule of solvent, in this case water, in m^3
        # self.entropic_unit = 136.6  # measured in MPa
        self.entropic_unit = K_B*T/V_m*1e-6  # measured in MPa
        print(f'Entropic unit = {self.entropic_unit:.2f} [MPa];', end='\n')
        #
        # self.G = 0.13               # measured in MPa
        # self.gamma = self.G/self.entropic_unit
        self.gamma = 0.001           # As in the simulations of Sect. 5 in Kang & Huang JMPS 2010
        # self.chi =  0.348
        self.chi = 0.4
        self.G = self.gamma*self.entropic_unit
        print(f'gamma=N*V_m = {self.gamma:.2e} [dimensionless];', end=' ')
        print(f'Shear modulus = N*K_B*T = {self.G:.2f} [MPa];', end=' ')
        print(f'Flory parameter = {self.chi:.3f} [dimensionless];', end='\n')
        #
        vapor_pressure = 3.2e-3    # 3.2 KPa, but measured in MPa
        self.p0_bar = vapor_pressure/(self.entropic_unit)
        print(f'Normalized vapor pressure = {self.p0_bar:.2e} [dimensionless];', end=' ')
        self.p_bar = self.p0_bar * np.exp(mu_bar)
        print(f'External solvent pressure = {self.p_bar*self.entropic_unit*1e3:.2f} [KPa];', end=' ')
        print(f'Normalized external pressure = {self.p_bar:.2e} [MPa];', end='\n')
        #        
        # self.density =  1.23 # measured in [g/mL]

        self.L = length      # measured in mm
        self.d = thickness    # measured in mm
        self.w = width # measured in mm

        self.filename_suffix = f'_phi0={self.phi0:.1f}_muBarAbs={np.abs(mu_bar):.6f}'
        print(f'Filename suffix: ' + self.filename_suffix, end='\n')

        def auxIsotropic(s):
            return s*self.dH(s*s*s) + self.gamma
        max_attempts = 100000
        attempts = 0
        lambda_initial = phi0*1.1
        while attempts< max_attempts:
            aux_value = auxIsotropic(lambda_initial)
            if aux_value>0:
                break
            lambda_initial+=0.01
            attempts+=1
        self.lambda_iso = scipy.optimize.fsolve(auxIsotropic, lambda_initial)[0]
        print(f'Isotropic extension: {self.lambda_iso:.3f}; lambda_initial = {lambda_initial}', end=' ')

        def auxUniaxial(s):
            return s*self.gamma + self.dH(s)
        max_attempts = 100000
        attempts = 0
        lambda_initial = phi0*1.1
        while attempts< max_attempts:
            aux_value = auxUniaxial(lambda_initial)
            if aux_value>0:
                break
            lambda_initial+=0.01
            attempts+=1
        self.lambda_target = scipy.optimize.fsolve(auxUniaxial, lambda_initial)[0]
        print(f'Uniaxial extension: {self.lambda_target:.3f}; lambda_initial = {lambda_initial}', end='\n')

        def auxEnergyDensity(lambda1, lambda2, lambda3):
            gel=self; phi0=gel.phi0; G=gel.G; chi=gel.chi; nu=gel.entropic_unit; gamma=gel.gamma; mu_bar=gel.mu_bar; p_bar=gel.p_bar
            J= lambda1*lambda2*lambda3
            phi = phi0/J
            return 0.5*G*(lambda1**2 + lambda2**2 + lambda3**2 - 3) + nu*((J-phi0)*np.log(1-phi) + phi0*chi*(1-phi) - gamma*log(J) + (p_bar - mu_bar)*(J-phi0) )

        lambda_iso = self.lambda_iso
        self.reference_energy_density = auxEnergyDensity(lambda_iso, lambda_iso, lambda_iso)                
        print(f'Energy density of isotropic expansion: {self.reference_energy_density:.5f}', end=' ')

    def phi(self, J):
        return self.phi0/J

    def H(self, J):
        return (J - self.phi0)*log(1-self.phi(J))  + self.phi0 * self.chi*(1-self.phi(J)) - self.gamma*log(J) + (self.p_bar - self.mu_bar)*(J-self.phi0)

    def dH(self, J):
        return self.phi(J) + np.log(1-self.phi(J)) + self.chi * self.phi(J)**2  - self.gamma/J +self.p_bar - self.mu_bar

    def Gfun(self, lamb):
        nu = self.entropic_unit
        return (-self.dH(lamb)/lamb)*nu
    
    #Agregado Abril 01 de 2026 15:27
    def mu_fun(self, lamb):
        """
        Calcula el mu_bar asociado a una deformación uniaxial lambda.
        Basado en la condición de equilibrio uniaxial (tipo eq. 3.6 Kang & Huang 2010)
        """
        J = lamb        
        # return self.p_bar + self.gamma/lamb - phi - np.log(1 - phi) - self.chi * phi**2
        return self.phi(J)  + np.log(1-self.phi(J)) + self.chi * self.phi(J)**2 - self.gamma/J + self.gamma*lamb  + self.p0_bar
        
    # energy density in [MPa]
    def W(self, F):
               
        J = Det(F)
        C_tensor = F.trans * F
        
        gel = self
        G = gel.G
        nu = gel.entropic_unit               
        
        reference_energy_density =  self.reference_energy_density
        
        return 0.5*G*(Trace(C_tensor) - 3) + nu*gel.H(J) - reference_energy_density

In [2]:
from ngsolve import x, y, z   # ✅ IMPORTANTE

class Solve_gel3d:
    def __init__(self, gel, order=1):
        self.gel = gel
        self.order = order
        self.start_time = datetime.datetime.now()  

    def add_mesh(self, mesh_file):        
        self.mesh = Mesh(mesh_file)
    
    def Space(self):
        self.fes = VectorH1(self.mesh, order=self.order, dirichlet="bonded|debonded")
        print('nDoF = {}'.format(self.fes.ndof))
        
    def model(self):
        u  = self.fes.TrialFunction()
        I = Id(self.mesh.dim)
        F = I + Grad(u)

        def negpart(var):
            return (sqrt(var**2)-var)*0.5        
        
        AA = 1e5

        # hydrogel model        
        self.a = BilinearForm(self.fes, symmetric=False)
        self.a += Variation(self.gel.W(F).Compile() * dx)

        # contacto (usa y directamente)
        self.a += Variation(AA*negpart(y+u[1])**2 * dx)
        
    def Solve_incremental_softening(self):
        self.Space()
        self.gfu = GridFunction(self.fes)

        # condición inicial uniaxial
        lambda_initial = 1.1

        if self.mesh.dim == 3:
            u0 = CoefficientFunction((0, (lambda_initial - 1.0)*y, 0))
        else:
            u0 = CoefficientFunction((0, (lambda_initial - 1.0)*y))

        self.gfu.Set(u0)

        # continuación en μ
        mu_bar_end = self.gel.mu_bar
        mu_bar_initial = self.gel.mu_fun(lambda_initial)
        nIterations = 15
        self.gel.mu_bar = Parameter (mu_bar_initial)
        self.model()

        lambda_list = np.linspace(lambda_initial, self.gel.lambda_target, nIterations)
        mu_list = [self.gel.mu_fun(la) for la in lambda_list]

        # final iteration
        mu_list.append(mu_bar_end)        

        filename = 'gridfunctions/result' + \
            self.gel.filename_suffix + \
            "_order={}".format(self.order)

        tol = 1e-3
        maxits = 100

        # indexes_iterations = range(nIterations+1)
        indexes_iterations = [0]
        for numIteration in indexes_iterations:

            mu_i = mu_list[numIteration]

            print("*** Iteration #", numIteration, ". mu_bar = ", mu_i)

            if numIteration == nIterations:
                tol = 1e-6
                maxits = 500

            self.gel.mu_bar.Set(mu_i)
            # self.gel.p_bar = self.gel.p0_bar * np.exp(mu_i)

            self.gfu, _, _ = SolveNonlinearMinProblem(
                a=self.a,
                gfu=self.gfu,
                maxits=maxits,
                tol=tol,
                alpha=1e-2
            )

            self.gfu.Save(filename + '_iter=' + str(numIteration).zfill(2) + '.gfu')

            print("Total time elapsed =", datetime.datetime.now() - self.start_time)

In [3]:
def SolveNonlinearMinProblem(a, gfu, tol=1e-08, maxits=50, alpha=1.0):
    
    start_time = datetime.datetime.now()  

    res = gfu.vec.CreateVector()
    du  = gfu.vec.CreateVector()
    w   = gfu.vec.CreateVector()  # para line search
    
    precond = 'bddc'
    c = Preconditioner(a, precond)

    for it in range(maxits):

        with TaskManager():
            # Residuo actual
            a.Apply(gfu.vec, res)

            # Ensamblar Jacobiano
            a.AssembleLinearization(gfu.vec)

            c.Update()
            inv = CGSolver(a.mat, c.mat, maxsteps=1000)

            du.data = inv * res

        # ==============================
        # 🔹 LINE SEARCH (backtracking)
        # ==============================
        step = alpha
        success = False

        res_norm_old = sqrt(abs(InnerProduct(res, res)))

        for ls in range(10):  # máximo 10 intentos

            w.data = gfu.vec - step * du

            with TaskManager():
                a.Apply(w, res)
                res_norm_new = sqrt(abs(InnerProduct(res, res)))

            if res_norm_new < res_norm_old:
                success = True
                break

            step *= 0.5  # reducir paso

        if not success:
            print("⚠️ Line search falló, usando paso pequeño")
        
        # actualizar solución
        gfu.vec.data = w

        stopcritval = sqrt(abs(InnerProduct(du, res)))

        print("Newton iteration:", it, "Time elapsed =", datetime.datetime.now() - start_time)
        print("Residual norm =", res_norm_new, "Step =", step)

        if stopcritval < tol:
            break

    return gfu, stopcritval, it

In [4]:
# data = [dummy, L, w, d, phi0, abs(mu_bar)]   We will suppose that mu_bar is negative
data = ['', '90', '15.0', '1.62', '1.00', '0.0916'] 

order = 1

mesh_file = 'meshes/mesh0.vol.gz'

L = float(data[1])
w = float(data[2])   # ✅ CORREGIDO
d = float(data[3])   # ✅ CORREGIDO
phi0 = float(data[4])
mu_bar = - float(data[5])

print(f'L={L}, w={w}, d={d}, phi0={phi0}, mu_bar={mu_bar}')

gel = gel_3D(length=L, width=w, thickness=d, phi0=phi0, mu_bar=mu_bar)

modelling = Solve_gel3d(gel, order=order)
modelling.add_mesh(mesh_file)

L=90.0, w=15.0, d=1.62, phi0=1.0, mu_bar=-0.0916
Initial polymer volume fraction = 1.00 [dimensionless]; Normalized chemical potential = -0.091600 [dimensionless?]; Entropic unit = 137.21 [MPa];
gamma=N*V_m = 1.00e-03 [dimensionless]; Shear modulus = N*K_B*T = 0.14 [MPa]; Flory parameter = 0.400 [dimensionless];
Normalized vapor pressure = 2.33e-05 [dimensionless]; External solvent pressure = 2.92 [KPa]; Normalized external pressure = 2.13e-05 [MPa];
Filename suffix: _phi0=1.0_muBarAbs=0.091600
Isotropic extension: 1.262; lambda_initial = 1.2700000000000002 Uniaxial extension: 2.000; lambda_initial = 2.0100000000000007
Energy density of isotropic expansion: -55.06968 

In [6]:
gfu_test = modelling.gfu

Draw(gfu_test, deformation=True)
                

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [5]:
modelling.Solve_incremental_softening()

nDoF = 90978
*** Iteration # 0 . mu_bar =  -1.158011620899397
Newton iteration: 0 Time elapsed = 0:00:01.343721
Residual norm = 0.19982682214518976 Step = 0.005
⚠️ Line search falló, usando paso pequeño
Newton iteration: 1 Time elapsed = 0:00:03.269795
Residual norm = 0.19982690472293516 Step = 9.765625e-06
⚠️ Line search falló, usando paso pequeño
Newton iteration: 2 Time elapsed = 0:00:05.164382
Residual norm = 0.1998269954869808 Step = 9.765625e-06
⚠️ Line search falló, usando paso pequeño
Newton iteration: 3 Time elapsed = 0:00:07.064062
Residual norm = 0.1998270944368859 Step = 9.765625e-06
⚠️ Line search falló, usando paso pequeño
Newton iteration: 4 Time elapsed = 0:00:08.972061
Residual norm = 0.19982720157210065 Step = 9.765625e-06
⚠️ Line search falló, usando paso pequeño
Newton iteration: 5 Time elapsed = 0:00:10.958055
Residual norm = 0.19982731689216865 Step = 9.765625e-06
⚠️ Line search falló, usando paso pequeño
Newton iteration: 6 Time elapsed = 0:00:13.571542
Residual 

In [ ]:
modelling.Space()
gfu_test = GridFunction(modelling.fes)
lambda_initial = 2.0

if modelling.mesh.dim == 3:
    u0 = CoefficientFunction((0, (lambda_initial - 1.0)*y, 0))
else:
    u0 = CoefficientFunction((0, (lambda_initial - 1.0)*y))

gfu_test.Set(u0)

Draw(gfu_test, deformation=True)